# MT-Bench and Chatbot Arena: Multi-Turn and Human Preference

Aggregated benchmark scores miss the qualities users actually care about: helpfulness, conversational flow, and judgment. MT-Bench and Chatbot Arena introduced evaluation paradigms centered on human preference rather than correctness against reference answers.

## MT-Bench: Multi-Turn Instruction Following

MT-Bench (Zheng et al. 2023) evaluated models on 80 multi-turn conversations across 8 categories: writing, roleplay, extraction, reasoning, math, coding, knowledge, and STEM. Each question had a follow-up turn designed to test whether models could handle context.

Key insight: single-turn instruction following is much easier than multi-turn. A model that gives a great first response often fails at follow-ups that require referencing or extending its previous answer. MT-Bench exposed this gap clearly.

GPT-4 as judge was used to score responses 1-10. This established LLM-as-judge as a practical evaluation method (the paper showed >80% agreement with human raters).

## Chatbot Arena and ELO Ratings

Chatbot Arena (Zheng et al. 2023, LMSYS) took a different approach: head-to-head human preference. Users sent real questions to two anonymous models and voted for the better response. With enough votes, ELO ratings could be computed.

ELO was borrowed from chess: each match updates both players' ratings based on the outcome and the expected probability. A win against a strong opponent gains more rating than a win against a weak one.

Arena became the most trusted public leaderboard for chat models because it used real user intent rather than curated benchmarks, and because the human preference signal is harder to game than static multiple-choice questions.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import random

@dataclass
class ArenaModel:
    name: str
    elo: float = 1000.0
    wins: int = 0
    losses: int = 0
    ties: int = 0

def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a) / 400))

def elo_update(model_a: ArenaModel, model_b: ArenaModel,
               outcome: str, k: float = 32.0):
    ea = expected_score(model_a.elo, model_b.elo)
    eb = expected_score(model_b.elo, model_a.elo)
    if outcome == 'a':
        sa, sb = 1.0, 0.0
        model_a.wins += 1; model_b.losses += 1
    elif outcome == 'b':
        sa, sb = 0.0, 1.0
        model_b.wins += 1; model_a.losses += 1
    else:
        sa, sb = 0.5, 0.5
        model_a.ties += 1; model_b.ties += 1
    model_a.elo += k * (sa - ea)
    model_b.elo += k * (sb - eb)

# Simulate arena matches
models = {
    'ModelA': ArenaModel('ModelA', elo=1000.0),
    'ModelB': ArenaModel('ModelB', elo=1000.0),
    'ModelC': ArenaModel('ModelC', elo=1000.0),
}

# True quality: A > B > C
true_quality = {'ModelA': 0.7, 'ModelB': 0.55, 'ModelC': 0.4}

rng = random.Random(42)
matchups = [('ModelA','ModelB'), ('ModelA','ModelC'), ('ModelB','ModelC')] * 50
rng.shuffle(matchups)

for ma_name, mb_name in matchups:
    ma, mb = models[ma_name], models[mb_name]
    qa, qb = true_quality[ma_name], true_quality[mb_name]
    r = rng.random()
    if r < qa / (qa + qb) * 0.8:
        outcome = 'a'
    elif r > 1 - qb / (qa + qb) * 0.8:
        outcome = 'b'
    else:
        outcome = 'tie'
    elo_update(ma, mb, outcome)

print(f'{'Model':<12} {'ELO':>8} {'W':>5} {'L':>5} {'T':>5}')
print('-' * 40)
for name, m in sorted(models.items(), key=lambda x: -x[1].elo):
    print(f'{name:<12} {m.elo:>8.1f} {m.wins:>5} {m.losses:>5} {m.ties:>5}')

## Multi-Turn Evaluation

MT-Bench evaluation requires tracking conversation state. The follow-up question is designed to be unanswerable without the correct context from the first response.

In [ ]:
@dataclass
class MTBenchEval:
    category: str
    turn1: str
    turn2: str
    model_name: str
    scores: list = field(default_factory=list)

    def run_suite(self, model_fn, judge_fn) -> dict:
        history = []
        results = []
        for turn_idx, question in enumerate([self.turn1, self.turn2]):
            if history:
                context = '\n'.join(f'Q: {h[0]}\nA: {h[1]}' for h in history)
                prompt = f'{context}\nQ: {question}'
            else:
                prompt = question
            response = model_fn(prompt)
            score = judge_fn(question, response)
            history.append((question, response))
            results.append({'turn': turn_idx+1, 'score': score, 'response': response[:100]})
        self.scores = [r['score'] for r in results]
        return {'category': self.category, 'turns': results, 'avg': sum(self.scores)/len(self.scores)}

# Mock model and judge
def simple_model(prompt):
    if 'fibonacci' in prompt.lower():
        return 'The Fibonacci sequence: 1, 1, 2, 3, 5, 8, 13, 21...'
    if 'sum' in prompt.lower() and 'first' in prompt.lower():
        return 'The sum of the first 10 Fibonacci numbers is 143.'
    return 'I would be happy to help with that.'

def mock_judge(question, response):
    return 7.5 if len(response) > 30 else 4.0

eval_session = MTBenchEval(
    category='math',
    turn1='Explain the Fibonacci sequence.',
    turn2='What is the sum of the first 10 numbers you listed?',
    model_name='simple-model'
)
results = eval_session.run_suite(simple_model, mock_judge)
print(f'Category: {results["category"]}')
print(f'Average score: {results["avg"]:.1f}')
for t in results['turns']:
    print(f'  Turn {t["turn"]}: {t["score"]} | {t["response"][:60]}...')

## Limitations of Arena Evaluation

Chatbot Arena is the best current proxy for real user preference, but it has known weaknesses:

**Selection bias**: users who visit the arena are not representative of all users. They tend to ask harder, more exploratory questions.

**Topic drift**: as models improve, users send harder prompts, which shifts the comparison baseline over time.

**Length and style bias**: even with human judges, verbosity and confident tone influence votes in ways that do not track accuracy.

**English-language skew**: the majority of votes come from English-speaking users, disadvantaging multilingual evaluation.

Despite these limitations, arena ELO has proven more predictive of real-world model quality than static benchmark scores.